# 第20章：组合理论

使用同期间年化小数收益与协方差，核对组合方差、GMV、A/B/C/D、切点及存在条件。解析式和第9章优化器共用相同输入。代码新生成two.png和frontier.png到RUN_DIR以核构造曲线；随后另显示两张已有构造参考图。两组图不是同一次生成。French两个历史窗口默认跳过，不产生新市场前沿。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch20_portfolio_theory_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys, tempfile

sys.path.insert(0, str(ROOT / 'code'))
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import ch20_portfolio_theory as workflow
from quant_course.portfolio_theory import (global_minimum_variance_weights, minimum_variance_target_weights,
                                           portfolio_moments, tangency_weights)
from quant_course.optimization_tools import solve_mean_variance
print('Project root:', ROOT)

## 1. 双重求和与二次型

逐项展开 \(\sum_i\sum_j w_iw_j\sigma_{ij}\) 并与 \(\mathbf{w}^\top\Sigma\mathbf{w}\) 比较（练习 20-E02 的数字）。

In [ ]:
vol = np.array([0.10, 0.20, 0.30])
corr = np.array([[1, 0.5, 0], [0.5, 1, -0.2], [0, -0.2, 1]])
cov = np.outer(vol, vol) * corr
w = np.array([0.5, 0.3, 0.2])
double_sum = sum(w[i] * w[j] * cov[i, j] for i in range(3) for j in range(3))
assert np.isclose(double_sum, w @ cov @ w) and np.isclose(double_sum, 0.01126)
print(round(double_sum, 6), round(np.sqrt(double_sum), 4))

## 2. 第 8 节手算：GMV、A/B/C/D、目标收益、切点

In [ ]:
two = workflow.two_asset_example()
assert np.allclose(two['gmv']['weights'], [11/14, 3/14])
k = two['constants']
assert np.isclose(k['B'] / k['A'], two['gmv']['mean']) and np.isclose(1 / k['A'], two['gmv']['variance'])
assert np.allclose(two['target_0.08']['weights'], [0.5, 0.5]) and np.isclose(two['target_0.08']['variance'], 0.025)
assert np.allclose(two['tangency']['weights'], [19/34, 15/34])
assert np.isclose(two['tangency']['sharpe'] ** 2, two['max_sharpe_squared_formula'])
{key: (round(v, 6) if isinstance(v, float) else v) for key, v in k.items()}, round(two['tangency']['sharpe'], 5)

## 3. 无效率分支与切点存在条件（20-E06、20-E07）

In [ ]:
mu = np.array([0.06, 0.10]); cov2 = np.array(two['covariance'])
low = minimum_variance_target_weights(mu, cov2, target_return=0.05)
mirror = minimum_variance_target_weights(mu, cov2, target_return=2 * k['B'] / k['A'] - 0.05)
assert np.allclose(low, [1.25, -0.25])
assert np.isclose(low @ cov2 @ low, mirror @ cov2 @ mirror)
assert k['B'] - 0.08 * k['A'] < 0
try:
    tangency_weights(mu, cov2, risk_free_return=0.08)
    raise AssertionError('expected rejection')
except ValueError as error:
    print('rf=0.08 rejected:', error)

## 4. 图与临界相关系数

临界相关 \(\sigma_1/\sigma_2=0.6\)：\(\rho=0.5\) 时长仓最低波动略低于 15%，\(\rho=1\) 时等于 15%。

In [ ]:
tmp = RUN_DIR
rows = workflow.plot_two_asset(tmp / 'two.png')['min_long_only_volatility_by_rho']
assert rows['+0.5'] < 0.15 and np.isclose(rows['+1.0'], 0.15)
three = workflow.plot_frontier_cml(tmp / 'frontier.png')
print({r: round(v, 4) for r, v in rows.items()})
print('3-asset GMV vol', round(three['gmv_volatility'], 4), 'tangency Sharpe', round(three['tangency_sharpe'], 4))

## 5. 默认禁用的French两窗口

历史方案会核对六个冻结文件的哈希，再比较正文第11节的样本内估计与权重剪裁。默认不调用此分支，表中历史数值不是本次构造计算。

In [ ]:
if RUN_FRENCH:
    from quant_course.french_frozen import load_industries_and_rf
    returns, rf, provenance = load_industries_and_rf()
    assert provenance['is_point_in_time'] is False
    summary, weights = workflow.french_windows(returns, rf)
    for label, row in summary.items():
        print(label, {k: round(v, 4) for k, v in row.items()})
    cov = returns.loc['2010-01':'2019-12'].cov().to_numpy()
    g = global_minimum_variance_weights(cov)
    clipped = np.clip(g, 0, None); clipped /= clipped.sum()
    qp = solve_mean_variance(np.zeros(10), cov / np.max(np.abs(cov))).weights
    vol = lambda x: float(np.sqrt(x @ cov @ x))
    assert vol(g) <= vol(qp) <= vol(clipped)
    print('2010s monthly vol: GMV %.3f%%, QP %.3f%%, clipped %.3f%%' % (100*vol(g), 100*vol(qp), 100*vol(clipped)))
    weights.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 6. 解读边界

历史两窗口的切点权重变化说明均值估计误差会被协方差逆放大，不提供样本外或成本后结论。滚动组合方法见Project 3；本Notebook当前仅计算构造矩。

## 图形读取

横轴为波动率，纵轴为同期间均值；任意给定矩下现金与风险基金的配置线是CAL。前面的计算在RUN_DIR生成two.png和frontier.png，本格另显示两张冻结参考图。French图只属于禁用分支的历史样本内矩描述，不代表样本外优势。

In [ ]:
from IPython.display import Image, display
for name in ["ch20_two_asset_correlation.png", "ch20_frontier_cml.png"]:
    display(Image(filename=str(ROOT / "fixtures/figures/a4" / name)))